# Spread Decomposition: AAPL (NASDAQ)

Effective spread = realized spread + price impact (Huang-Stoll identity).
This decomposition is naturally from the maker's perspective: effective spread
is revenue, price impact is the cost of adverse selection, and realized spread
is what the maker keeps.

Data: LOBSTER sample, AAPL, 2012-06-21 (the only freely available date).

In [ ]:
import polars as pl
from fetch_data import fetch_lobster_sample, prepare_lobster

import markoutlib as mo

## Download and prepare data

In [ ]:
messages, orderbook = fetch_lobster_sample("AAPL", level=10)
trades, quotes = prepare_lobster(messages, orderbook)

print(f"Trades: {trades.shape[0]:,}")
print(f"Quotes: {quotes.shape[0]:,}")
trades.head()

## Compute markouts at multiple horizons

We compute markouts from the maker's perspective. The spread
decomposition methods use these markouts internally.

In [ ]:
result = mo.compute(
    trades=trades,
    quotes=quotes,
    horizons=mo.seconds(1, 5, 10, 30, 60),
    unit="bps",
    perspective="maker",
)

## Spread decomposition at 5 seconds

At the 5-second horizon, how much of the maker's spread revenue
survives adverse selection?

In [ ]:
result.spread_decomposition(horizon=mo.seconds(5))

The effective spread is 0.68 bps — consistent with AAPL's tight quoted spread on NASDAQ. At 5 seconds, the realized spread is essentially zero: the maker's entire spread revenue has been consumed by price impact (0.68 bps). Five seconds is the breakeven horizon for passive market-making in this name.

## Verify the Huang-Stoll identity

Effective spread should equal realized spread plus price impact exactly.
This is an algebraic identity, not an approximation.

In [ ]:
decomp = result.spread_decomposition(horizon=mo.seconds(5))
for row in decomp.iter_rows(named=True):
    eff = row["effective_spread_mean"]
    real = row["realized_spread_mean"]
    imp = row["price_impact_mean"]
    print(f"Effective: {eff:.4f}, Realized: {real:.4f}, Impact: {imp:.4f}")
    print(
        f"Identity check: {eff:.4f} = {real + imp:.4f}"
        f" (diff: {abs(eff - real - imp):.2e})"
    )

## Decomposition across horizons

How does the split between realized spread and price impact evolve?
At short horizons the maker keeps some spread; at longer horizons
adverse selection erodes it.

In [ ]:
rows = []
for h in [1, 5, 10, 30, 60]:
    d = result.spread_decomposition(horizon=mo.seconds(h))
    for r in d.iter_rows(named=True):
        rows.append(
            {
                "horizon_s": h,
                "effective_spread": r["effective_spread_mean"],
                "realized_spread": r["realized_spread_mean"],
                "price_impact": r["price_impact_mean"],
            }
        )

decomp_df = pl.DataFrame(rows)
print(decomp_df)

At 1 second the maker still retains 0.13 bps of realized spread (20% of effective). By 5 seconds it's gone. At 10 seconds price impact actually exceeds the effective spread (0.70 vs 0.68 bps), pushing the realized spread slightly negative — the maker is underwater on a mark-to-market basis. This pattern is typical of liquid large-cap equities where information is incorporated quickly.

## Segment by trade size

Do larger trades impose more adverse selection on the maker?

In [ ]:
q50 = trades["size"].quantile(0.5)
trades_tagged = trades.with_columns(
    pl.when(pl.col("size") <= q50)
    .then(pl.lit("small"))
    .otherwise(pl.lit("large"))
    .alias("size_bucket")
)

result_sized = mo.compute(
    trades=trades_tagged,
    quotes=quotes,
    horizons=mo.seconds(5),
    unit="bps",
    perspective="maker",
)

result_sized.spread_decomposition(horizon=mo.seconds(5), by="size_bucket")

Small trades have a wider effective spread (0.75 vs 0.60 bps) and higher price impact (0.75 vs 0.61 bps). The realized spread is near zero for both buckets at 5 seconds — confirming that the breakeven horizon is consistent regardless of trade size. The wider effective spread on small trades comes from executing further from the mid, while the higher price impact suggests small trades in AAPL carry slightly more information per trade.